In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Jawaharlal_Nehru_Stadium_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,250.0,NaN,133.0,69.0,69.0,59.0,43.0,55.0,51.0,84.0,358.0,362.0
1,2,356.0,193.0,215.0,88.0,61.0,80.0,39.0,66.0,44.0,88.0,NaN,345.0
2,3,395.0,NaN,NaN,131.0,79.0,83.0,NaN,NaN,41.0,77.0,484.0,300.0
3,4,349.0,222.0,91.0,75.0,70.0,114.0,121.0,76.0,40.0,96.0,398.0,305.0
4,5,330.0,236.0,112.0,95.0,131.0,124.0,63.0,65.0,55.0,112.0,450.0,309.0
5,6,413.0,246.0,108.0,108.0,207.0,78.0,58.0,71.0,NaN,174.0,394.0,290.0
6,7,378.0,272.0,147.0,114.0,128.0,227.0,NaN,82.0,62.0,182.0,364.0,271.0
7,8,398.0,119.0,190.0,133.0,112.0,118.0,54.0,92.0,46.0,154.0,406.0,306.0
8,9,462.0,208.0,73.0,152.0,147.0,123.0,61.0,103.0,32.0,143.0,429.0,NaN
9,10,429.0,176.0,164.0,143.0,163.0,109.0,NaN,117.0,19.0,NaN,314.0,307.0


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   31 non-null     float64
 3   March      32 non-null     float64
 4   April      34 non-null     float64
 5   May        35 non-null     float64
 6   June       33 non-null     float64
 7   July       23 non-null     float64
 8   August     30 non-null     float64
 9   September  33 non-null     float64
 10  October    35 non-null     float64
 11  November   34 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,250.0,183.806452,133.00000,69.0,69.0,59.0,43.000000,55.000000,51.0,84.0,358.000000,362.0
1,2,356.0,193.000000,215.00000,88.0,61.0,80.0,39.000000,66.000000,44.0,88.0,298.205882,345.0
2,3,395.0,183.806452,117.09375,131.0,79.0,83.0,49.782609,68.633333,41.0,77.0,484.000000,300.0
3,4,349.0,222.000000,91.00000,75.0,70.0,114.0,49.782609,76.000000,40.0,96.0,398.000000,305.0
4,5,330.0,236.000000,112.00000,95.0,131.0,124.0,63.000000,65.000000,55.0,112.0,450.000000,309.0


In [9]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
